# Evaluate Classic SPL Retention Model - Other Short

Comparing two models:
- **B-2760706**: Includes PUP (088) and Condo (078)
- **B-2779253**: Excludes PUP and Condo

In [ ]:
ENV_FOR_DYNACONF='prod'
DYNACONF_GIT_BRANCH='feature/B-2760706'

In [ ]:
import functools
import json
import operator
import os

import ltv_helpers.non_spark_helpers as nsh
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from classic_spl_ltv.config.paths import paths as p
from classic_spl_ltv.jobs.retention.pipeline import load_model
from sklearn.metrics import (
    average_precision_score,
    log_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

In [ ]:
pd.options.display.max_columns = 500
sns.set_style("ticks", rc={"figure.figsize": (14, 8)})
plt.rcParams["figure.figsize"] = (14, 8)

In [ ]:
model_line = "other"
model_type = "short"

## Load Models

In [ ]:
model_path_with = "tmx-smsiweb/classic-specialty-ltv/prod/feature/B-2760706/retention/models/other_short_ret.mdl"
model_path_without = "tmx-smsiweb/classic-specialty-ltv/prod/feature/B-2779253/retention/models/other_short_ret.mdl"

bst_with = load_model(model_path_with)
bst_without = load_model(model_path_without)

## Load Scored Data

In [ ]:
data_path_with = "tmx-smsiweb/classic-specialty-ltv/prod/feature/B-2760706/retention/data/scored/other_short_test_data_scored/"
data_path_without = "tmx-smsiweb/classic-specialty-ltv/prod/feature/B-2779253/retention/data/scored/other_short_test_data_scored/"

df_with = nsh.read_parquet_s3_to_pandas(data_path_with)
df_without = nsh.read_parquet_s3_to_pandas(data_path_without)

df_with["variant"] = "With PUP/Condo"
df_without["variant"] = "Without PUP/Condo"

In [ ]:
print(f"With PUP/Condo: {len(df_with):,} records")
print(f"Without PUP/Condo: {len(df_without):,} records")
print(f"Difference: {len(df_with) - len(df_without):,} records")

## Metrics Comparison

In [ ]:
def compute_metrics(df, label):
    y_true = df["term_ind"]
    y_score = df["model_prediction"]
    y_pred = np.column_stack((y_score, 1 - y_score))
    return {
        "variant": label,
        "records": len(df),
        "term_rate": y_true.mean(),
        "mean_prediction": y_score.mean(),
        "log_loss": log_loss(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_score),
        "avg_precision": average_precision_score(y_true, y_score),
    }

metrics = pd.DataFrame([
    compute_metrics(df_with, "With PUP/Condo"),
    compute_metrics(df_without, "Without PUP/Condo"),
])
metrics

## Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8), dpi=100)

for df, label, color in [(df_with, "With PUP/Condo", "blue"), (df_without, "Without PUP/Condo", "red")]:
    precision, recall, _ = precision_recall_curve(df["term_ind"], df["model_prediction"])
    ax.plot(recall, precision, label=label, color=color)

no_skill = df_with["term_ind"].mean()
ax.plot([0, 1], [no_skill, no_skill], linestyle="--", color="gray", label="No Skill")
ax.set_title("Precision-Recall Curve")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
plt.show()

## ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8), dpi=100)

for df, label, color in [(df_with, "With PUP/Condo", "blue"), (df_without, "Without PUP/Condo", "red")]:
    fpr, tpr, _ = roc_curve(df["term_ind"], df["model_prediction"])
    auc = roc_auc_score(df["term_ind"], df["model_prediction"])
    ax.plot(fpr, tpr, label=f"{label} (AUC={auc:.4f})", color=color)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="No Skill")
ax.set_title("ROC Curve")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
plt.show()

## Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=100, sharey=True)

axes[0].hist(df_with["model_prediction"], bins=50, edgecolor="black", color="blue", alpha=0.7)
axes[0].set_title("With PUP/Condo")
axes[0].set_xlabel("model_prediction")
axes[0].set_ylabel("Count")

axes[1].hist(df_without["model_prediction"], bins=50, edgecolor="black", color="red", alpha=0.7)
axes[1].set_title("Without PUP/Condo")
axes[1].set_xlabel("model_prediction")

plt.suptitle("Score Distribution Comparison")
plt.tight_layout()
plt.show()

## Term Rate by Line

In [ ]:
combined = pd.concat([df_with, df_without])

line_comparison = combined.groupby(["variant", "ply_line_cd"]).agg(
    count=("term_ind", "count"),
    actual_term_rate=("term_ind", "mean"),
    predicted_term_rate=("model_prediction", "mean"),
).reset_index()
line_comparison["diff"] = line_comparison["actual_term_rate"] - line_comparison["predicted_term_rate"]
line_comparison

## Univariate Plots

In [ ]:
category_cols = df_with.select_dtypes(include=["object"]).columns.to_list()
category_cols = ["nt6"] + [
    col for col in category_cols
    if "zip" not in col and "release_day_pol" not in col and col != "variant"
]

In [ ]:
df_with.nt6.value_counts(dropna=False)

In [ ]:
category_cols

In [ ]:
for col in category_cols:
    print(f"\n--- {col} (With PUP/Condo) ---")
    g1 = df_with.groupby(col).agg(
        count=("term_ind", "count"),
        actual_term_rate=("term_ind", "mean"),
        predicted_term_rate=("model_prediction", "mean"),
    ).reset_index()
    g1["diff"] = g1["actual_term_rate"] - g1["predicted_term_rate"]
    print(g1.to_string(index=False))

    print(f"\n--- {col} (Without PUP/Condo) ---")
    g2 = df_without.groupby(col).agg(
        count=("term_ind", "count"),
        actual_term_rate=("term_ind", "mean"),
        predicted_term_rate=("model_prediction", "mean"),
    ).reset_index()
    g2["diff"] = g2["actual_term_rate"] - g2["predicted_term_rate"]
    print(g2.to_string(index=False))

## SHAP Feature Importance

In [ ]:
sample_size = min(5000, len(df_with), len(df_without))

In [ ]:
feature_cols_with = bst_with.feature_name()
X_with = df_with.sample(n=sample_size, random_state=42)[feature_cols_with]

explainer_with = shap.TreeExplainer(bst_with)
shap_values_with = explainer_with.shap_values(X_with)

print("With PUP/Condo")
shap.summary_plot(shap_values_with, X_with, max_display=20)

In [ ]:
feature_cols_without = bst_without.feature_name()
X_without = df_without.sample(n=sample_size, random_state=42)[feature_cols_without]

explainer_without = shap.TreeExplainer(bst_without)
shap_values_without = explainer_without.shap_values(X_without)

print("Without PUP/Condo")
shap.summary_plot(shap_values_without, X_without, max_display=20)

## Summary

In [ ]:
metrics